# 09 - Agentes con LangChain

## Curso de LLMs y Aplicaciones de IA

**Duración estimada:** 2.5-3 horas

---

## Índice

1. [Tipos de Agentes en LangChain](#tipos)
2. [Herramientas integradas](#herramientas)
3. [Agente con RAG (Retriever Tool)](#rag)
4. [Memoria en Agentes](#memoria)
5. [Salidas estructuradas](#estructuradas)
6. [Ejercicios prácticos](#ejercicios)

---

## Objetivos de aprendizaje

Al finalizar este notebook, serás capaz de:
- Usar diferentes tipos de agentes de LangChain
- Integrar herramientas de búsqueda y RAG
- Añadir memoria conversacional al agente
- Obtener salidas estructuradas con Pydantic

In [1]:
# Install required libraries
#!pip install -q langchain langchain-groq langchain-community langchain-huggingface
#!pip install -q faiss-cpu sentence-transformers

In [2]:
import os
from getpass import getpass
import warnings
warnings.filterwarnings('ignore')

if 'GROQ_API_KEY' not in os.environ:
    os.environ['GROQ_API_KEY'] = getpass("Introduce tu GROQ API Key: ")

from langchain_groq import ChatGroq
llm = ChatGroq(model_name="llama-3.3-70b-versatile", temperature=0)
print("LLM configurado ✓")

Introduce tu GROQ API Key:  ········


LLM configurado ✓


<a name="tipos"></a>
## 1. Tipos de Agentes en LangChain

| Tipo | Descripción | Uso |
|------|-------------|-----|
| **Tool Calling** | Usa function calling nativo | GPT-4, Llama 3 |
| **ReAct** | Thought-Action-Observation | General |
| **Structured Chat** | Para inputs estructurados | Múltiples parámetros |

<a name="herramientas"></a>
## 2. Herramientas Integradas

LangChain proporciona muchas herramientas pre-construidas.

In [3]:
from langchain.tools import tool
from langchain_core.tools import Tool
from datetime import datetime
import random

# Custom tools
@tool
def calculator(expression: str) -> str:
    """Evalúa expresiones matemáticas. Ejemplo: '2+2' o '10*5'"""
    try:
        return str(eval(expression))
    except:
        return "Error en la expresión"

@tool
def get_datetime() -> str:
    """Obtiene fecha y hora actual."""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

@tool
def random_number(max_value: str) -> str:
    """Genera número aleatorio entre 1 y max_value."""
    try:
        return str(random.randint(1, int(max_value)))
    except:
        return "Error: proporciona un número"

@tool
def text_stats(text: str) -> str:
    """Devuelve estadísticas del texto: caracteres, palabras, oraciones."""
    chars = len(text)
    words = len(text.split())
    sentences = text.count('.') + text.count('!') + text.count('?')
    return f"Caracteres: {chars}, Palabras: {words}, Oraciones: {sentences}"

tools = [calculator, get_datetime, random_number, text_stats]
print(f"{len(tools)} herramientas configuradas")

4 herramientas configuradas


In [4]:
from langchain_classic.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate

# Agent prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", """Eres un asistente con acceso a herramientas.
Usa las herramientas cuando sea necesario.
Siempre responde en español."""),
    ("placeholder", "{chat_history}"),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}")
])

agent = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

print("Agente creado ✓")

Agente creado ✓


In [5]:
# Test agent
result = agent_executor.invoke({
    "input": "Genera un número aleatorio del 1 al 100 y multiplícalo por 7",
    "chat_history": []
})
print(f"\nRespuesta: {result['output']}")



> Entering new AgentExecutor chain...

Invoking: `random_number` with `{'max_value': '100'}`


10
Invoking: `calculator` with `{'expression': '(resultado del random_number) * 7'}`


Error en la expresión
Invoking: `random_number` with `{'max_value': '100'}`
responded: Lo siento, no puedo generar un número aleatorio y multiplicarlo por 7 en una sola respuesta. Primero, debo generar el número aleatorio y luego multiplicarlo por 7.



65
Invoking: `calculator` with `{'expression': '65 * 7'}`


455El número aleatorio generado es 65 y multiplicado por 7 es 455.

> Finished chain.

Respuesta: El número aleatorio generado es 65 y multiplicado por 7 es 455.


<a name="rag"></a>
## 3. Agente con RAG (Retriever Tool)

Combinemos un agente con acceso a una base de conocimiento.

In [6]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_classic.tools.retriever import create_retriever_tool

# Create knowledge base
company_docs = [
    Document(page_content="TechCorp ofrece tres planes: Básico (99€), Pro (299€), Enterprise."),
    Document(page_content="Horario de soporte: Lunes a Viernes, 9:00-18:00."),
    Document(page_content="Email soporte: soporte@techcorp.es. Tel: 900 123 456."),
    Document(page_content="Política devoluciones: 30 días para reembolso completo."),
    Document(page_content="TechCorp tiene oficinas en Madrid, Barcelona y Valencia."),
]

# Create vector store
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(company_docs, embeddings)
retriever = vectorstore.as_retriever()

# Create retriever tool
retriever_tool = create_retriever_tool(
    retriever,
    "company_search",
    "Busca información sobre TechCorp: precios, horarios, contacto, políticas."
)

print("Retriever tool creado ✓")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Retriever tool creado ✓


In [7]:
# Agent with RAG + other tools
rag_tools = [retriever_tool, calculator, get_datetime]

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", """Eres un asistente de TechCorp.
Usa company_search para preguntas sobre la empresa.
Usa calculator para cálculos.
Responde siempre en español."""),
    ("placeholder", "{chat_history}"),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}")
])

rag_agent = create_tool_calling_agent(llm, rag_tools, rag_prompt)
rag_executor = AgentExecutor(agent=rag_agent, tools=rag_tools, verbose=True)

print("Agente RAG creado ✓")

Agente RAG creado ✓


In [8]:
# Test RAG agent
result = rag_executor.invoke({
    "input": "¿Cuánto cuesta el plan Básico y cuánto pagaría por 6 meses?",
    "chat_history": []
})
print(f"\nRespuesta: {result['output']}")



> Entering new AgentExecutor chain...

Invoking: `company_search` with `{'query': 'precio plan Básico TechCorp'}`


TechCorp ofrece tres planes: Básico (99€), Pro (299€), Enterprise.

TechCorp tiene oficinas en Madrid, Barcelona y Valencia.

Email soporte: soporte@techcorp.es. Tel: 900 123 456.

Política devoluciones: 30 días para reembolso completo.
Invoking: `calculator` with `{'expression': 'precio_plan_basico * 6'}`


Error en la expresión
Invoking: `calculator` with `{'expression': '99 * 6'}`


594El plan Básico cuesta 99€ al mes. Por 6 meses, pagarías 594€.

> Finished chain.

Respuesta: El plan Básico cuesta 99€ al mes. Por 6 meses, pagarías 594€.


<a name="memoria"></a>
## 4. Memoria en Agentes

In [9]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# Session store
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

# Agent with memory
agent_with_memory = RunnableWithMessageHistory(
    rag_executor,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history"
)

print("Agente con memoria configurado ✓")

Agente con memoria configurado ✓


C:\Users\jassa\anaconda3\Lib\site-packages\IPython\core\interactiveshell.py:3699: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [10]:
# Test memory
config = {"configurable": {"session_id": "user1"}}

# First question
r1 = agent_with_memory.invoke({"input": "Hola, me llamo Ana"}, config=config)
print(f"R1: {r1['output']}\n")

# Follow-up
r2 = agent_with_memory.invoke({"input": "¿Cuál es el horario de soporte?"}, config=config)
print(f"R2: {r2['output']}\n")

# Test memory
r3 = agent_with_memory.invoke({"input": "¿Cómo me llamo?"}, config=config)
print(f"R3: {r3['output']}")



> Entering new AgentExecutor chain...
¡Hola Ana! Me alegra conocerte. ¿En qué puedo ayudarte hoy? ¿Tienes alguna pregunta sobre TechCorp o necesitas ayuda con algo más?

> Finished chain.
R1: ¡Hola Ana! Me alegra conocerte. ¿En qué puedo ayudarte hoy? ¿Tienes alguna pregunta sobre TechCorp o necesitas ayuda con algo más?



> Entering new AgentExecutor chain...

Invoking: `company_search` with `{'query': 'horario de soporte de TechCorp'}`


TechCorp tiene oficinas en Madrid, Barcelona y Valencia.

Email soporte: soporte@techcorp.es. Tel: 900 123 456.

TechCorp ofrece tres planes: Básico (99€), Pro (299€), Enterprise.

Horario de soporte: Lunes a Viernes, 9:00-18:00.El horario de soporte de TechCorp es de lunes a viernes de 9:00 a 18:00. ¿Necesitas ayuda con algo más?

> Finished chain.
R2: El horario de soporte de TechCorp es de lunes a viernes de 9:00 a 18:00. ¿Necesitas ayuda con algo más?



> Entering new AgentExecutor chain...

Invoking: `company_search` with `{'query': 'horario

<a name="estructuradas"></a>
## 5. Salidas Estructuradas

In [11]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser

# Define output structure
class TaskAnalysis(BaseModel):
    task: str = Field(description="La tarea identificada")
    difficulty: str = Field(description="Dificultad: fácil, media, difícil")
    estimated_time: str = Field(description="Tiempo estimado")
    tools_needed: list = Field(description="Herramientas necesarias")

parser = PydanticOutputParser(pydantic_object=TaskAnalysis)

print("Parser configurado")
print(parser.get_format_instructions())

Parser configurado
The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"task": {"description": "La tarea identificada", "title": "Task", "type": "string"}, "difficulty": {"description": "Dificultad: fácil, media, difícil", "title": "Difficulty", "type": "string"}, "estimated_time": {"description": "Tiempo estimado", "title": "Estimated Time", "type": "string"}, "tools_needed": {"description": "Herramientas necesarias", "items": {}, "title": "Tools Needed", "type": "array"}}, "required": ["task", "difficulty", "estimated_time", "tools_needed"]}
```


In [12]:
# Use structured output
structured_prompt = f"""Analiza la siguiente tarea y responde en el formato especificado.

{parser.get_format_instructions()}

Tarea: Crear un dashboard de ventas que muestre gráficos interactivos y se actualice en tiempo real."""

response = llm.invoke(structured_prompt)
print(response.content)

Para cumplir con el esquema JSON proporcionado, debemos crear un objeto que contenga las propiedades "task", "difficulty", "estimated_time" y "tools_needed", con sus respectivos valores. A continuación, te presento un ejemplo de cómo podría ser el objeto JSON que describe la tarea de crear un dashboard de ventas:

```json
{
  "task": "Crear un dashboard de ventas que muestre gráficos interactivos y se actualice en tiempo real",
  "difficulty": "difícil",
  "estimated_time": "2 semanas",
  "tools_needed": [
    "Tableau",
    "Power BI",
    "Python",
    "Librerías de visualización de datos",
    "Base de datos relacional"
  ]
}
```

En este ejemplo, hemos identificado la tarea como "Crear un dashboard de ventas que muestre gráficos interactivos y se actualice en tiempo real". La dificultad se ha clasificado como "difícil" debido a la complejidad de crear un dashboard interactivo que se actualice en tiempo real. El tiempo estimado para completar esta tarea se ha establecido en "2 seman

<a name="ejercicios"></a>
## 6. Ejercicios Prácticos

### Ejercicio 1: Agente de soporte técnico

In [13]:
# Exercise 1: Create a tech support agent
# Add documents about common tech problems
# Create appropriate tools
# Test with realistic support questions

In [15]:
# ============================================================
# Exercise 1: Create a tech support agent
# ============================================================

from langchain.tools import tool
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_classic.tools.retriever import create_retriever_tool
from langchain_classic.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate

from datetime import datetime
import random


# ============================================================
# 1. Base de conocimiento: documentos de soporte técnico
# ============================================================

support_docs = [
    Document(page_content="""
    Problema: No funciona el WiFi.
    Pasos recomendados:
    1. Comprobar si otros dispositivos tienen conexión.
    2. Reiniciar el router durante 30 segundos.
    3. Verificar que el modo avión esté desactivado.
    4. Olvidar la red WiFi y volver a conectarse.
    5. Si el problema afecta a varios usuarios, abrir ticket de soporte.
    """),

    Document(page_content="""
    Problema: No puedo acceder a la VPN.
    Posibles causas:
    - Contraseña incorrecta.
    - Usuario bloqueado por varios intentos fallidos.
    - Cliente VPN desactualizado.
    - Problema temporal del servidor VPN.
    Solución:
    Actualizar el cliente VPN, comprobar usuario y contraseña,
    verificar la autenticación multifactor y revisar el estado del servicio VPN.
    """),

    Document(page_content="""
    Problema: El correo electrónico no envía mensajes.
    Pasos recomendados:
    1. Revisar conexión a internet.
    2. Comprobar si el mensaje está en la bandeja de salida.
    3. Revisar si el archivo adjunto supera los 25 MB.
    4. Reiniciar Outlook o el cliente de correo.
    5. Si continúa fallando, abrir ticket con captura del error.
    """),

    Document(page_content="""
    Problema: He olvidado mi contraseña.
    Solución:
    Usar el portal de restablecimiento de contraseña.
    La nueva contraseña debe tener al menos 12 caracteres,
    incluir mayúsculas, minúsculas, números y símbolos.
    Si la cuenta está bloqueada, esperar 15 minutos o contactar con soporte.
    """),

    Document(page_content="""
    Problema: La impresora no imprime.
    Pasos recomendados:
    1. Comprobar que la impresora esté encendida.
    2. Revisar papel, tinta o tóner.
    3. Verificar que sea la impresora predeterminada.
    4. Vaciar la cola de impresión.
    5. Reinstalar el controlador si el problema continúa.
    """),

    Document(page_content="""
    Problema: El ordenador va muy lento.
    Posibles soluciones:
    - Reiniciar el equipo.
    - Cerrar aplicaciones innecesarias.
    - Comprobar espacio libre en disco.
    - Ejecutar análisis antivirus.
    - Revisar actualizaciones pendientes.
    Si el equipo sigue lento tras estos pasos, abrir ticket.
    """),

    Document(page_content="""
    Problema: Pérdida de archivos.
    Recomendaciones:
    1. No seguir modificando carpetas afectadas.
    2. Revisar la papelera de reciclaje.
    3. Revisar el historial de versiones de OneDrive o SharePoint.
    4. Contactar con soporte si los archivos no aparecen.
    Este caso debe tratarse como prioridad alta.
    """),

    Document(page_content="""
    Niveles de prioridad:
    - Crítica: afecta a toda la empresa o impide trabajar a muchos usuarios. SLA: 2 horas.
    - Alta: afecta a un usuario pero bloquea su trabajo. SLA: 4 horas.
    - Media: afecta parcialmente al trabajo. SLA: 24 horas.
    - Baja: consulta o petición menor. SLA: 48 horas.
    """)
]


# ============================================================
# 2. Crear vector store y retriever tool
# ============================================================

support_embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

support_vectorstore = FAISS.from_documents(
    support_docs,
    support_embeddings
)

support_retriever = support_vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

support_search_tool = create_retriever_tool(
    support_retriever,
    "support_search",
    "Busca información sobre problemas técnicos comunes: WiFi, VPN, correo, contraseña, impresora, lentitud del ordenador y pérdida de archivos."
)

print("Base de conocimiento y retriever de soporte creados ✓")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Base de conocimiento y retriever de soporte creados ✓


In [16]:
# ============================================================
# 3. Herramientas personalizadas de soporte técnico
# ============================================================

@tool
def check_service_status(service_name: str) -> str:
    """
    Comprueba el estado simulado de un servicio técnico.
    Input: nombre del servicio, por ejemplo 'VPN', 'correo', 'wifi' o 'impresora'.
    """
    service = service_name.lower()

    status = {
        "vpn": "Incidencia parcial detectada. Algunos usuarios pueden tener problemas de conexión.",
        "correo": "Servicio operativo. No hay incidencias generales registradas.",
        "email": "Servicio operativo. No hay incidencias generales registradas.",
        "wifi": "Servicio operativo. Si falla, probablemente sea un problema local del dispositivo o router.",
        "impresora": "Servicio operativo. Revisar cola de impresión, papel, tóner y controlador.",
        "onedrive": "Servicio operativo. Revisar historial de versiones si faltan archivos.",
        "sharepoint": "Servicio operativo. Revisar permisos y sincronización."
    }

    for key, value in status.items():
        if key in service:
            return f"Estado de {service_name}: {value}"

    return f"No tengo información específica sobre el servicio '{service_name}'. Recomiendo abrir un ticket si el problema continúa."


@tool
def create_support_ticket(description: str) -> str:
    """
    Crea un ticket simulado de soporte técnico.
    Input: descripción breve del problema del usuario.
    """
    ticket_id = f"INC-{datetime.now().strftime('%Y%m%d')}-{random.randint(1000, 9999)}"

    return (
        f"Ticket creado correctamente.\n"
        f"ID: {ticket_id}\n"
        f"Descripción: {description}\n"
        f"Estado: Abierto\n"
        f"Prioridad inicial: Pendiente de clasificación por soporte."
    )


@tool
def classify_priority(problem_description: str) -> str:
    """
    Clasifica la prioridad de un problema técnico.
    Input: descripción del problema.
    """
    text = problem_description.lower()

    if any(word in text for word in ["toda la empresa", "nadie puede", "caído", "no funciona para nadie", "servidor caído"]):
        return "Prioridad crítica. Afecta a muchos usuarios o a un servicio completo. SLA estimado: 2 horas."

    if any(word in text for word in ["no puedo trabajar", "bloqueado", "vpn", "archivos perdidos", "pérdida de archivos"]):
        return "Prioridad alta. El problema bloquea el trabajo del usuario. SLA estimado: 4 horas."

    if any(word in text for word in ["lento", "impresora", "correo", "wifi"]):
        return "Prioridad media. El problema afecta al trabajo, pero puede tener solución temporal. SLA estimado: 24 horas."

    return "Prioridad baja. Consulta o incidencia menor. SLA estimado: 48 horas."


support_tools = [
    support_search_tool,
    check_service_status,
    create_support_ticket,
    classify_priority
]

print(f"{len(support_tools)} herramientas de soporte configuradas ✓")

4 herramientas de soporte configuradas ✓


In [17]:
# ============================================================
# 4. Crear agente de soporte técnico
# ============================================================

support_prompt = ChatPromptTemplate.from_messages([
    ("system", """
    Eres un agente de soporte técnico de primer nivel.

    Tu trabajo es:
    - Ayudar al usuario con problemas técnicos comunes.
    - Usar support_search para buscar soluciones en la base de conocimiento.
    - Usar check_service_status si el usuario pregunta por el estado de un servicio.
    - Usar classify_priority si hay que valorar la gravedad del problema.
    - Usar create_support_ticket si el problema no se resuelve o parece importante.

    Responde siempre en español.
    Da pasos claros y ordenados.
    No inventes datos técnicos que no estén en la base de conocimiento.
    Si el problema parece grave, recomienda abrir ticket.
    """),
    ("placeholder", "{chat_history}"),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}")
])

support_agent = create_tool_calling_agent(
    llm,
    support_tools,
    support_prompt
)

support_executor = AgentExecutor(
    agent=support_agent,
    tools=support_tools,
    verbose=True
)

print("Agente de soporte técnico creado ✓")

Agente de soporte técnico creado ✓


In [18]:
# ============================================================
# 5. Test con preguntas realistas de soporte
# ============================================================

test_questions = [
    "No puedo conectarme al WiFi de la oficina. ¿Qué hago?",
    "No me funciona la VPN y necesito trabajar ya. ¿Hay alguna incidencia?",
    "He olvidado mi contraseña y creo que mi cuenta está bloqueada.",
    "Outlook no me deja enviar correos con un archivo adjunto grande.",
    "He perdido unos archivos importantes de OneDrive. ¿Qué prioridad tiene esto?"
]

for question in test_questions:
    print("=" * 80)
    print(f"Pregunta: {question}")
    print("-" * 80)

    result = support_executor.invoke({
        "input": question,
        "chat_history": []
    })

    print(f"Respuesta:\n{result['output']}\n")

Pregunta: No puedo conectarme al WiFi de la oficina. ¿Qué hago?
--------------------------------------------------------------------------------


> Entering new AgentExecutor chain...

Invoking: `support_search` with `{'query': 'problemas de conexión WiFi'}`



    Problema: No funciona el WiFi.
    Pasos recomendados:
    1. Comprobar si otros dispositivos tienen conexión.
    2. Reiniciar el router durante 30 segundos.
    3. Verificar que el modo avión esté desactivado.
    4. Olvidar la red WiFi y volver a conectarse.
    5. Si el problema afecta a varios usuarios, abrir ticket de soporte.
    


    Problema: El correo electrónico no envía mensajes.
    Pasos recomendados:
    1. Revisar conexión a internet.
    2. Comprobar si el mensaje está en la bandeja de salida.
    3. Revisar si el archivo adjunto supera los 25 MB.
    4. Reiniciar Outlook o el cliente de correo.
    5. Si continúa fallando, abrir ticket con captura del error.
    


    Problema: El ordenador va muy lento

## Resumen

En este notebook hemos aprendido:

1. **Tipos de agentes**: Tool calling, ReAct, Structured
2. **Herramientas**: Crear y usar herramientas personalizadas
3. **RAG + Agentes**: Combinar retrieval con capacidad de acción
4. **Memoria**: Mantener contexto entre turnos
5. **Salidas estructuradas**: Obtener datos parseables

En el siguiente notebook veremos **LangGraph** para crear flujos de trabajo más complejos.

---

## Referencias

- [LangChain Agents](https://python.langchain.com/docs/modules/agents/)
- [LangChain Tools](https://python.langchain.com/docs/modules/tools/)

In [14]:
import session_info
session_info.show(html = False)

-----
ipykernel                   6.31.0
langchain_classic           1.0.7
langchain_community         0.4.2
langchain_core              1.4.0
langchain_groq              1.1.2
langchain_huggingface       NA
pandas                      2.3.3
pydantic                    2.12.4
session_info                v1.0.1
-----
IPython             9.7.0
jupyter_client      8.6.3
jupyter_core        5.8.1
jupyterlab          4.4.7
notebook            7.4.5
-----
Python 3.13.9 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 19:09:58) [MSC v.1929 64 bit (AMD64)]
Windows-10-10.0.19045-SP0
-----
Session information updated at 2026-06-10 19:47
